# Multi-FLORT Determinism Experiment: Does It Hold Across Sensor Models?

Earlier experiments established that OOI's M2M delivery is **content-deterministic
but not byte-deterministic** for one FLORT (NSIF, `flort_datalogger`) and, as a
control, one CTD (`ctdbp_datalogger`). This notebook stress-tests that result
across **different fluorometer models on different platforms with different
processors**, to see whether determinism holds up or a variant behaves
chaotically.

**Instruments tested** (three FLORT variants, from the OOI example scripts):

| Model | Platform | Processor |
|---|---|---|
| `FLORTD` | NSIF surface mooring (`CE02SHSM`) | `flort_datalogger` (burst) |
| `FLORTK` | Wire-following profiler (`CE09OSPM`) | `flort_wfp` |
| `FLORTJ` | Coastal surface-piercing profiler (`CE02SHSP`) | `flort_cspp` |

**Design.** Rather than one notebook per instrument (as before), everything is
driven by a single **instrument registry** and one shared harness: define the
sensors once, run the same fresh-M2M determinism test over each, and compare the
results side by side. Adding another sensor later is one registry entry.

**Two questions:**
1. *Determinism* — for each model, do repeated identical M2M requests return the
   same content (content hash) even if the file bytes differ?
2. *Structure* — how much do the three models differ in schema (variables and
   dtypes), given their different platforms and processors?

## Setup

Fresh-request M2M functions, the three FLORT processors, and hashing. `time` and
`hashlib` support the retry logic and the byte/content hashing.


In [1]:
%matplotlib inline
import os
import time
import hashlib
import numpy as np
import pandas as pd
import xarray as xr

from ooi_data_explorations.common import (m2m_request, m2m_collect,
                                          list_deployments, get_deployment_dates)
from ooi_data_explorations.uncabled.process_flort import (flort_datalogger,
                                                          flort_wfp, flort_cspp)

OUT_DIR = os.path.join(os.getcwd(), 'flort_multi_experiment')
os.makedirs(OUT_DIR, exist_ok=True)
print('Saving experiment outputs to:', OUT_DIR)

Saving experiment outputs to: /home/jovyan/code/ooi-data-explorations/python/examples/notebooks/flort_multi_experiment


## The instrument registry

Each entry fully specifies one sensor: its reference designator, delivery method
and stream, its processor and the keyword arguments that processor takes, which
deployment to sample, and how long a window to request. The `deploy_index` and
`window_hours` values follow the OOI example scripts; profilers use a longer
window because they sample fewer times per hour than the fixed-depth NSIF sensor.

To add a sensor, append a dict. To drop one, remove it. Nothing else changes.


In [2]:
INSTRUMENTS = [
    {   # fixed-depth NSIF fluorometer
        'name': 'FLORTD_NSIF',
        'site': 'CE02SHSM', 'node': 'RID27', 'sensor': '02-FLORTD000',
        'method': 'recovered_host', 'stream': 'flort_sample',
        'processor': flort_datalogger, 'proc_kwargs': {'burst': True},
        'deploy_index': 0,   'window_hours': 24,
    },
    {   # wire-following profiler fluorometer
        'name': 'FLORTK_WFP',
        'site': 'CE09OSPM', 'node': 'WFP01', 'sensor': '04-FLORTK000',
        'method': 'recovered_wfp', 'stream': 'flort_sample',
        'processor': flort_wfp, 'proc_kwargs': {'grid': False},
        'deploy_index': -4,  'window_hours': 72,
    },
    {   # coastal surface-piercing profiler fluorometer
        'name': 'FLORTJ_CSPP',
        'site': 'CE02SHSP', 'node': 'SP001', 'sensor': '07-FLORTJ000',
        'method': 'recovered_cspp', 'stream': 'flort_sample',
        'processor': flort_cspp, 'proc_kwargs': {},
        'deploy_index': 10,  'window_hours': 48,
    },
]

N_REPEATS = 3          # fresh M2M requests per instrument; keep small (slow)
FLORT_TAG = r'.*FLORT.*\.nc$'

print(f'{len(INSTRUMENTS)} instruments registered; {N_REPEATS} repeats each.')

3 instruments registered; 3 repeats each.


## Shared harness (defined once)

The functions the previous notebooks duplicated live here a single time:

- `pick_window` — turn a registry entry into a concrete request window using the
  chosen deployment's dates.
- `request_flort` — one fresh asynchronous M2M request, with retries.
- `fingerprint` / `file_sha256` / `content_hash` — schema table, byte hash, and
  data+schema hash.
- `save_and_fingerprint` — save a dataset and return its fingerprint plus a
  metadata row (record count and both hashes).


In [3]:
def _iso(ts):
    '''Normalize any timestamp to an ISO-8601 UTC string M2M accepts.'''
    ts = pd.Timestamp(ts)
    if ts.tzinfo is not None:
        ts = ts.tz_convert('UTC').tz_localize(None)
    return ts.strftime('%Y-%m-%dT%H:%M:%S.000Z')


def pick_window(inst):
    '''Anchor a window inside the chosen deployment (1 day in, window_hours long).'''
    deps = list_deployments(inst['site'], inst['node'], inst['sensor'])
    if not deps:
        raise RuntimeError(f"no deployments found for {inst['name']}")
    deploy = deps[inst['deploy_index']]
    start, _stop = get_deployment_dates(inst['site'], inst['node'],
                                        inst['sensor'], deploy)
    a = pd.Timestamp(start) + pd.Timedelta(days=1)
    b = a + pd.Timedelta(hours=inst['window_hours'])
    return _iso(a), _iso(b), deploy


def request_flort(inst, start, stop, retries=3, pause=30):
    '''One fresh asynchronous M2M request -> xarray Dataset, with retries.'''
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            r = m2m_request(inst['site'], inst['node'], inst['sensor'],
                            inst['method'], inst['stream'], start, stop)
            if not r:
                raise RuntimeError('empty request response')
            ds = m2m_collect(r, FLORT_TAG)
            if ds is None or ds.time.size == 0:
                raise RuntimeError('no data returned')
            return ds
        except Exception as e:
            last_err = e
            print(f'    attempt {attempt}/{retries} failed: {e}')
            if attempt < retries:
                time.sleep(pause)
    raise RuntimeError(f'request failed after {retries} attempts: {last_err}')


def fingerprint(ds, label):
    '''Per-variable schema table: dtype, dims, shape.'''
    rows = []
    for name, da in ds.variables.items():
        rows.append({'label': label, 'variable': name,
                     'dtype': str(da.dtype),
                     'dims': ','.join(da.dims),
                     'shape': 'x'.join(str(s) for s in da.shape)})
    return pd.DataFrame(rows).sort_values('variable').reset_index(drop=True)


def file_sha256(path):
    '''SHA-256 of a file's raw bytes (byte-identity test).'''
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def content_hash(ds):
    '''Hash of variable values + schema only, ignoring file-level metadata.'''
    h = hashlib.sha256()
    for name in sorted(ds.variables):
        da = ds[name]
        h.update(name.encode())
        h.update(str(da.dtype).encode())
        h.update(','.join(da.dims).encode())
        h.update(np.ascontiguousarray(da.values).tobytes())
    return h.hexdigest()


def save_and_fingerprint(ds, inst_name, rep, stage):
    '''Save to NetCDF; return (fingerprint_df, meta_row).'''
    label = f'{inst_name}_rep{rep}_{stage}'
    path = os.path.join(OUT_DIR, f'{label}.nc')
    ds.to_netcdf(path, mode='w', format='NETCDF4', engine='h5netcdf')
    fp = fingerprint(ds, label)
    fp['instrument'], fp['rep'], fp['stage'] = inst_name, rep, stage
    meta = {'instrument': inst_name, 'rep': rep, 'stage': stage, 'label': label,
            'n_records': int(ds.time.size), 'n_vars': len(ds.data_vars),
            'file_sha256': file_sha256(path), 'content_hash': content_hash(ds)}
    return fp, meta

## Run the experiment

For each registered instrument we resolve its window once, then issue `N_REPEATS`
fresh M2M requests, saving and fingerprinting the raw result and the processed
result each time. A failure on one instrument (bad designator, empty window,
processor error) is reported and skipped so the rest still run.

> **Slow.** Every repetition is a fresh async request. Three instruments x three
> repeats is nine requests plus processing; expect several minutes. Drop
> `N_REPEATS` to 2 for a quick shakeout.


In [4]:
fingerprints, records, status = [], [], []

for inst in INSTRUMENTS:
    name = inst['name']
    print(f'=== {name} ===')
    try:
        start, stop, deploy = pick_window(inst)
        print(f'  deployment {deploy}, window {start} -> {stop}')
    except Exception as e:
        print(f'  SKIP (window resolution failed): {e}')
        status.append({'instrument': name, 'status': f'window failed: {e}'})
        continue

    n_ok = 0
    for rep in range(1, N_REPEATS + 1):
        try:
            raw = request_flort(inst, start, stop)
        except Exception as e:
            print(f'  rep {rep}: request failed: {e}')
            continue
        raw = raw.sortby('time').drop_duplicates(dim='time')

        fp, meta = save_and_fingerprint(raw, name, rep, 'raw')
        fingerprints.append(fp); records.append(meta)

        try:
            proc = inst['processor'](raw, **inst['proc_kwargs'])
            fp, meta = save_and_fingerprint(proc, name, rep, 'proc')
            fingerprints.append(fp); records.append(meta)
        except Exception as e:
            print(f'  rep {rep}: processing failed: {type(e).__name__}: {e}')

        n_ok += 1
        time.sleep(5)

    status.append({'instrument': name, 'status': f'{n_ok}/{N_REPEATS} reps ok'})

all_fp = pd.concat(fingerprints, ignore_index=True) if fingerprints else pd.DataFrame()
meta_df = pd.DataFrame(records)
print('\nRun status:')
print(pd.DataFrame(status).to_string(index=False))

=== FLORTD_NSIF ===
  deployment 1, window 2015-04-03T20:15:00.000Z -> 2015-04-04T20:15:00.000Z


Requesting:
	refdes: CE02SHSM-RID27-02-FLORTD000
	method: recovered_host
	stream: flort_sample
	from 2015-04-03T20:15:00.000Z to 2015-04-04T20:15:00.000Z
Waiting for OOINet to process and prepare data request, this may take up to 20 minutes.


Waiting:   0%|          | 0/400 [00:00<?, ?it/s]

Merging the data files into a single dataset


Requesting:
	refdes: CE02SHSM-RID27-02-FLORTD000
	method: recovered_host
	stream: flort_sample
	from 2015-04-03T20:15:00.000Z to 2015-04-04T20:15:00.000Z
Waiting for OOINet to process and prepare data request, this may take up to 20 minutes.


...

Merging the data files into a single dataset
Creating and adding a profile variable to the data set ...


Requesting:
	refdes: CE02SHSP-SP001-07-FLORTJ000
	method: recovered_cspp
	stream: flort_sample
	from 2018-09-30T23:20:25.000Z to 2018-10-02T23:20:25.000Z
Waiting for OOINet to process and prepare data request, this may take up to 20 minutes.


Waiting:   0%|          | 0/400 [00:00<?, ?it/s]

Merging the data files into a single dataset
Creating and adding a profile variable to the data set ...



Run status:
 instrument      status
FLORTD_NSIF 3/3 reps ok
 FLORTK_WFP 3/3 reps ok
FLORTJ_CSPP 3/3 reps ok


## Analysis 1 - Determinism per model (the primary question)

For each instrument and stage, how many distinct file hashes and content hashes did
the repeated requests produce? `content_identical = True` (one content hash across
all reps) means that model is content-deterministic; `byte_identical` is expected
to be `False` because each M2M file carries fresh metadata. If any model shows
`content_identical = False`, that is the "chaotic" outcome we are looking for.


In [5]:
det = (meta_df.groupby(['instrument', 'stage'])
       .agg(n_reps=('file_sha256', 'size'),
            distinct_file=('file_sha256', 'nunique'),
            distinct_content=('content_hash', 'nunique'),
            record_counts=('n_records', lambda s: sorted(set(s))))
       .reset_index())
det['content_identical'] = det['distinct_content'] == 1
det['byte_identical']    = det['distinct_file'] == 1
det

,instrument,stage,n_reps,distinct_file,distinct_content,record_counts,content_identical,byte_identical
0,FLORTD_NSIF,proc,3,3,1,[96],True,False
1,FLORTD_NSIF,raw,3,3,1,[14275],True,False
2,FLORTJ_CSPP,proc,3,3,1,[759],True,False
3,FLORTJ_CSPP,raw,3,3,1,[759],True,False
4,FLORTK_WFP,proc,3,3,1,[8163],True,False
5,FLORTK_WFP,raw,3,3,1,[8163],True,False


In [6]:
# Per-dataset hashes (truncated), so any difference is visible by instrument/rep
(meta_df[['instrument', 'rep', 'stage', 'n_records', 'file_sha256', 'content_hash']]
 .assign(file_sha256=lambda d: d['file_sha256'].str[:10],
         content_hash=lambda d: d['content_hash'].str[:10])
 .sort_values(['instrument', 'stage', 'rep'])
 .reset_index(drop=True))

,instrument,rep,stage,n_records,file_sha256,content_hash
0,FLORTD_NSIF,1,proc,96,f49863ead6,f6da42881b
1,FLORTD_NSIF,2,proc,96,ca8d1e83df,f6da42881b
2,FLORTD_NSIF,3,proc,96,38b1f3d0d6,f6da42881b
3,FLORTD_NSIF,1,raw,14275,5411ee966c,da8f2eb0a1
4,FLORTD_NSIF,2,raw,14275,a4a7dcac8c,da8f2eb0a1
5,FLORTD_NSIF,3,raw,14275,ee55b276b8,da8f2eb0a1
6,FLORTJ_CSPP,1,proc,759,e398c3e31a,c6de97f60d
7,FLORTJ_CSPP,2,proc,759,ebb4678dd9,c6de97f60d
8,FLORTJ_CSPP,3,proc,759,9477eb9832,c6de97f60d
9,FLORTJ_CSPP,1,raw,759,ec7e73d6b3,e836fde7a2


## Analysis 2 - Schema differences across models

Determinism asks whether a model is self-consistent across requests. This asks how
much the three *models* differ from each other, given their different platforms and
processors. We compare the processed form of the first repetition of each
instrument: which variables each model has, and where shared variables differ in
dtype. Large differences here are expected (a profiler carries depth/pressure a
fixed-depth sensor does not) and are a structural finding, not a determinism issue.


In [7]:
if not all_fp.empty:
    sig = all_fp[(all_fp['rep'] == 1) & (all_fp['stage'] == 'proc')]

    presence = (sig.assign(present=True)
                .pivot_table(index='variable', columns='instrument',
                             values='present', aggfunc='any', fill_value=False))
    print('Variables NOT shared by all models:')
    not_shared = presence[~presence.all(axis=1)]
    print(not_shared if len(not_shared) else '  (all variables shared)')

    print('\nDtype by model (shared variables):')
    dtypes = sig.pivot_table(index='variable', columns='instrument',
                             values='dtype', aggfunc='first')
    shared_dtypes = dtypes.dropna()
    display(shared_dtypes)
else:
    print('No data collected - nothing to compare.')

Variables NOT shared by all models:
instrument                                  FLORTD_NSIF  FLORTJ_CSPP  \
variable                                                               
beta_700_burst_stats                               True        False   
depth                                             False         True   
estimated_chlorophyll_burst_stats                  True        False   
fluorometric_cdom_burst_stats                      True        False   
fluorometric_cdom_qartod_executed                 False         True   
fluorometric_chlorophyll_a_qartod_executed        False         True   
lat                                               False         True   
lon                                               False         True   
optical_backscatter_qartod_executed               False         True   
profile                                           False         True   
raw_internal_temp                                  True         True   
sea_water_pressure          

instrument,FLORTD_NSIF,FLORTJ_CSPP,FLORTK_WFP
variable,,,
bback,float64,float64,float64
bback_qc_executed,float64,uint8,uint8
bback_qc_results,float64,uint8,uint8
bback_qc_summary_flag,float64,int32,int32
beta_700,float64,float64,float64
beta_700_qc_executed,float64,uint8,uint8
beta_700_qc_results,float64,uint8,uint8
beta_700_qc_summary_flag,float64,int32,int32
deployment,float64,int32,int32


## Findings

Three FLORT models were tested across three platforms with three processors, at
`N_REPEATS = 3` fresh M2M requests each. All three returned data (the CSPP after
switching to a mid-history deployment). Two conclusions:

**1. Determinism holds across every model.** All six (instrument × stage) groups
were **content-identical but not byte-identical** — each group's three repetitions
produced three different file hashes but a single content hash:

| Model | raw content hash | proc content hash | records (raw / proc) |
|---|---|---|---|
| FLORTD_NSIF | `da8f2eb0a1` | `f6da42881b` | 14275 / 96 |
| FLORTK_WFP | `0901944d11` | `435af48248` | 8163 / 8163 |
| FLORTJ_CSPP | `e836fde7a2` | `c6de97f60d` | 759 / 759 |

Record counts were also stable across repetitions. Since this now holds for three
different fluorometer models and processors (`flort_datalogger`, `flort_wfp`,
`flort_cspp`) — on top of the earlier CTD control — the content-deterministic
behavior is a property of the **M2M pipeline**, not of any one sensor or processor.
No model behaved chaotically.

A processor detail visible in the counts: the NSIF sensor collapses from 14,275 raw
to 96 processed because `flort_datalogger(burst=True)` burst-averages, while the two
profiler processors reorganize profiles without burst-averaging, so their raw and
processed record counts match.

**2. The models differ structurally — by platform and by processor.** The
variables that are not shared by all three split cleanly into two explainable
groups:

- **Platform-driven (profilers only, FLORTK + FLORTJ):** `depth`, `profile`,
  `lat`, `lon`, `sea_water_pressure` and its QC variables. A profiler moves through
  the water column, so it must carry depth/pressure/position; the fixed-depth NSIF
  sensor does not.
- **Processor-driven (NSIF only, FLORTD):** `beta_700_burst_stats`,
  `estimated_chlorophyll_burst_stats`, `fluorometric_cdom_burst_stats`, `stats`.
  These exist only because `flort_datalogger(burst=True)` computes burst statistics;
  the profiler processors do not burst-average, so they have none.

These differences are structural, not determinism problems — each model is
internally reproducible; they describe different things because they *are*
different instruments.

## Summary

This notebook generalized the FLORT determinism test across three fluorometer
models on three platforms with three processors (NSIF `FLORTD` /
`flort_datalogger`, wire-following profiler `FLORTK` / `flort_wfp`, coastal
surface-piercing profiler `FLORTJ` / `flort_cspp`), driven by a single instrument
registry and one shared harness.

**Main Result.** Determinism held for every model: repeated identical M2M requests
returned content-identical (though not byte-identical) results in all six
instrument × stage groups. The behavior is pipeline-general. The models differ in
schema, but every difference traces to either the platform (profilers carry
depth/pressure/position) or the processor (the NSIF sensor carries burst
statistics) — not to any non-determinism.

**Reproducibility.** All datasets and their file/content hashes are saved in
`flort_multi_experiment/`.